# 09 · Vision Transformer
*Cortar la imagen en patches, tratarlos como tokens y dejar que la atención decida qué mirar*

Notebook del episodio 9 de **Visión: de convoluciones a ViT** ([ver la lección](https://d32vh5r3pvn9et.cloudfront.net/cursos/vision/vision-transformer)). Es el mismo guion del video, en texto, con cada bloque de código para correr y su salida real.

**Mecanismo:** patches 4×4: 64 tokens de 48 → patch embedding = conv kernel = stride = 4 (4e-7) → CLS + posiciones aprendidas: 65 × 128 → ViT chico (6 bloques, 4 heads, 811.146 parámetros vs. 272.474 de ResNet-20) → atención a mano = PyTorch (1,5e-7); logits del CLS reales → mapa de atención del CLS sobre un gato: casi nada a la tela de abajo → 30 epochs (2.000 imágenes de test): ResNet-20 84,2 % / ViT 71,6 %; con aug 89,1 % / 80,3 % → embeddings posicionales aprendieron la vecindad: coseno vecino 0,84, esquina −0,62 (sin entrenar −0,13 / −0,05) → paper ViT: algunos puntos debajo de ResNet en ImageNet, con 14M–300M imágenes gana, 88,55 %

**Cómo correrlo:** con el entorno de `~/Movies/serie-vision/lab` (`cd lab && uv run --project ~/Movies/serie-alineacion/lab --group notebooks --with jupyterlab jupyter lab ../notebooks`, o en VS Code eligiendo el kernel `lab/.venv`). Todo corre offline: Gemma 2 2B sale de la caché de Hugging Face, y los entrenamientos son de juguete, con datos sintéticos.

Cada bloque de código arranca de cero (`%reset -f`): la celda de **preparación** es lo que el video no muestra, y la siguiente es exactamente el código que aparece en pantalla.

## ▶ Antes de empezar

Corré la celda de abajo una vez: baja el código de la serie y sus dependencias. Funciona en Google Colab (gratis) y en cualquier Jupyter. Para guardar tus cambios en Colab: **Archivo → Guardar una copia en Drive**.

In [ ]:
import os, subprocess, sys
SERIE = "vision"
REPO = "https://github.com/Faridmurzone/paperclase-notebooks"

def _raiz():  # carpeta del repo si ya estamos adentro
    d = os.path.abspath(os.getcwd())
    while True:
        if os.path.isfile(os.path.join(d, ".paperclase-notebooks")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

RAIZ = _raiz()
if RAIZ is None:
    RAIZ = os.path.abspath("paperclase-notebooks")
    if not os.path.isdir(RAIZ):
        subprocess.run(["git", "clone", "--depth", "1", REPO, RAIZ], check=True)
os.chdir(os.path.join(RAIZ, SERIE, "lab"))  # los bloques de código importan desde snippets/ y leen datos/
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements.txt"], check=True)
os.environ["HF_HUB_OFFLINE"] = "0"  # el notebook original corría sin red; acá baja los modelos de Hugging Face
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("listo:", os.getcwd())

In [1]:
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import warnings; warnings.filterwarnings("ignore")

*Visión, de las convoluciones a los transformers. Episodio nueve: el ViT, Vision Transformer. Hasta acá, toda red que vimos asume que un píxel tiene que ver con sus vecinos: la convolución está construida así. El ViT tira esa suposición. Corta la imagen en cuadraditos, los trata como las palabras de una frase, y aprende solo qué mirar. Hoy lo armamos pieza por pieza y lo comparamos con la ResNet.*

### Una imagen como una frase

- Cortar en patches de 4×4: 64 tokens
- Cada patch, una capa lineal: un vector de 128
- Un token extra, CLS, y la posición sumada
- Atención: cada token mira a todos los demás

Una imagen de treinta y dos por treinta y dos se corta en patches de cuatro por cuatro: ocho por ocho, sesenta y cuatro cuadraditos. Cada uno se aplana y pasa por una capa lineal: un vector de ciento veintiocho números, un token. Se agrega un token extra, el CLS, class token, que no viene de la imagen y al final resume todo; y a cada token se le suma un vector que dice dónde está. Después, bloques transformer: en cada uno, todos los tokens se miran entre sí con atención, sin importar si están al lado o en la otra punta.

## Parte 1 · Patches como tokens

El primer paso es puramente de forma: pasar de una grilla de píxeles a una secuencia.

### El camino de una imagen por el ViT

El recorrido completo. La imagen, tres canales por treinta y dos por treinta y dos, se corta en sesenta y cuatro patches; cada uno tiene tres por cuatro por cuatro, cuarenta y ocho números. Una capa lineal los lleva a ciento veintiocho, se agrega el CLS adelante y se suman las posiciones: sesenta y cinco tokens. Eso se llama patch embedding. Seis bloques transformer, cada uno con atención y un MLP, multilayer perceptron, la red densa de dos capas que procesa cada token por separado. Y para clasificar, solo se usa la salida del CLS, con una capa lineal a diez clases.

imagen 3×32×32 → 64 patches de 48 → lineal 48→128 → + CLS, + posición → 6 bloques → salida del CLS → lineal 128→10

### Cortar y proyectar

Con una imagen real del test. unfold corta el alto en tiras de cuatro, y después el ancho: queda una grilla de ocho por ocho bloques. Reordenamos para tener sesenta y cuatro filas de cuarenta y ocho números. La forma habitual de escribir el patch embedding es una convolución con kernel cuatro y stride cuatro: el kernel nunca se superpone, así que cada salida mira un solo patch. Y lo comparamos con multiplicar los patches por la matriz de pesos de esa convolución. Sesenta y cuatro tokens de cuarenta y ocho; después, de ciento veintiocho, y la diferencia entre los dos caminos es cuatro por diez a la menos siete: son la misma cuenta.

In [2]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep09_patches.py
__file__ = 'snippets/ep09_patches.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import os, sys
import torch
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from _cifar import cargar, normalizar
X, _ = cargar("test", n=1)
x = normalizar(X)                            # (1, 3, 32, 32)
torch.manual_seed(0)

In [3]:
p, d = 4, 128
patches = x.unfold(2, p, p).unfold(3, p, p)          # (1, 3, 8, 8, 4, 4)
patches = patches.permute(0, 2, 3, 1, 4, 5).reshape(1, 64, 3 * p * p)
print("patches:", tuple(patches.shape))              # 64 tokens de 48

proy = torch.nn.Conv2d(3, d, kernel_size=p, stride=p)   # patch embedding
tok_conv = proy(x).flatten(2).transpose(1, 2)        # (1, 64, 128)
W = proy.weight.reshape(d, -1)                       # (128, 48)
tok_lineal = patches @ W.T + proy.bias               # una capa lineal
print("tokens:", tuple(tok_conv.shape),
      " dif. máx:", f"{(tok_conv - tok_lineal).abs().max().item():.1e}")

patches: (1, 64, 48)
tokens: (1, 64, 128)  dif. máx: 4.2e-07


### Patch embedding y posiciones

$$
N = \frac{H}{p}\cdot\frac{W}{p} = \frac{32}{4}\cdot\frac{32}{4} = 64
$$
<small>p: lado del patch; H, W: alto y ancho</small>

$$
z_i = E\, x_i + b,\quad E \in \mathbb{R}^{128 \times 48}
$$
<small>xᵢ: el patch i aplanado, 48 números</small>

$$
Z_0 = [\,x_{\text{CLS}};\ z_1;\ \dots;\ z_{64}\,] + E_{\text{pos}}
$$
<small>Eₚₒₛ: 65 × 128, una fila por posición, aprendida</small>

La cantidad de tokens: el alto sobre el lado del patch, por el ancho sobre el lado del patch. Treinta y dos sobre cuatro es ocho, y ocho por ocho, sesenta y cuatro. Cada patch aplanado, x sub i, de cuarenta y ocho números, se multiplica por una matriz E de ciento veintiocho por cuarenta y ocho, la misma para todos los patches. En el dibujo, una fila de E por el patch llena una coordenada del token. Apilamos el CLS y los sesenta y cuatro tokens, y les sumamos E pos, una tabla de sesenta y cinco filas aprendida. En el dibujo, la posición cero, del CLS, la uno y la dos eligen su fila. Sin esa suma, el ViT no tendría forma de saber qué patch está arriba y cuál abajo.

### El modelo entero, por formas

El ViT chico que vamos a entrenar. Patches de cuatro, tokens de ciento veintiocho, seis bloques y cuatro heads de atención. Con un batch de dos imágenes, los tokens con CLS son dos por sesenta y cinco por ciento veintiocho. Los seis bloques mantienen esa forma: cada bloque suma su resultado al token, como el bloque residual del episodio siete. Y la clasificación sale del token cero, el CLS, después de una layer norm, la normalización de cada token. Ochocientos once mil parámetros, contra doscientos setenta y dos mil de la ResNet de veinte capas: tres veces más.

In [4]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep09_modelo.py
__file__ = 'snippets/ep09_modelo.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import os, sys
import torch
AQUI = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, AQUI)
from _resnet import ViTChica, RedCifar, contar
torch.manual_seed(0)
x = torch.randn(2, 3, 32, 32)

In [5]:
vit = ViTChica(patch=4, d=128, capas=6, heads=4)
t = vit.patch_embed(x).flatten(2).transpose(1, 2)
print("patch embedding:", tuple(t.shape))          # 64 tokens de 128
z = vit.tokens(x)                                  # + CLS + posiciones
print("con CLS:        ", tuple(z.shape))
z = vit.encoder(z)                                 # 6 bloques transformer
print("después:        ", tuple(z.shape))
print("logits del CLS: ", tuple(vit.fc(vit.norm(z[:, 0])).shape))
print("parámetros ViT:", contar(vit), "| ResNet-20:", contar(RedCifar(n=3, residual=True)))

patch embedding: (2, 64, 128)
con CLS:         (2, 65, 128)
después:         (2, 65, 128)
logits del CLS:  (2, 10)
parámetros ViT: 811146 | ResNet-20: 272474


## Parte 2 · Atención entre patches

Cada token decide a qué otros tokens mirar, y con cuánto peso.

### Atención en tres pasos

$$
Q = Z W_Q,\quad K = Z W_K,\quad V = Z W_V
$$
<small>Z: 65 × 128; cada W: 128 × 32 por head</small>

$$
A = \mathrm{softmax}\!\Big(\frac{Q K^{\top}}{\sqrt{d_h}}\Big)
$$
<small>dₕ = 32; A: 65 × 65, cada fila suma 1</small>

$$
\text{fila del CLS: } \mathrm{softmax}(q_0 \cdot k_j / \sqrt{32})
$$
<small>cuánto mira el CLS a cada patch j</small>

$$
\text{salida}_0 = \textstyle\sum_j A_{0j}\, v_j
$$
<small>promedio de los valores, pesado por la atención</small>

Z son los sesenta y cinco tokens, cada uno de ciento veintiocho números. Tres matrices de pesos los proyectan en queries, keys y values: Q, K y V. Cada head usa treinta y dos dimensiones. El producto de Q por K transpuesta compara cada query con cada key: una matriz de sesenta y cinco por sesenta y cinco. En el dibujo, la fila cero, la del CLS, por la columna de un patch llena una celda. Se divide por raíz de treinta y dos, y el softmax convierte cada fila en pesos que suman uno. En el dibujo, logits reales del CLS en el último bloque para seis patches elegidos, tres altos y tres bajos, con el softmax calculado solo entre esos seis. Y la salida es el promedio de los values con esos pesos.

### La atención, a mano y contra PyTorch

Lo mismo en código, sobre el ViT ya entrenado. PyTorch guarda las tres matrices pegadas en una sola: la cortamos en tres. Multiplicamos los tokens normalizados por W Q, sumamos el sesgo, y partimos las ciento veintiocho dimensiones en cuatro heads de treinta y dos. Lo mismo para K. Softmax de Q por K transpuesta sobre raíz de treinta y dos. La matriz es de cuatro por sesenta y cinco por sesenta y cinco, y la diferencia con lo que calcula el módulo de PyTorch es uno coma cinco por diez a la menos siete. Cada fila suma uno. La fila del CLS, sin su propia columna, se reacomoda como una grilla de ocho por ocho: un número por patch.

**Cada línea, su término:**

- `Wq, Wk, Wv = capa.self_attn.in_proj_weight.chunk(3)` → $W_Q, W_K, W_V$ — las tres matrices, guardadas juntas
- `Q = (h @ Wq.T + bq).view(65, H, dh).transpose(0, 1)` → $Q = Z W_Q$ — partido en 4 heads de 32
- `A = F.softmax(Q @ K.transpose(1, 2) / dh ** 0.5, dim=-1)` → $A = \mathrm{softmax}(QK^\top/\sqrt{d_h})$ — 65 × 65 por head
- `cls = A[:, 0, 1:].mean(0).view(8, 8)` → $A_{0,1..64} \to 8 \times 8$ — la fila del CLS, como imagen

In [6]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep09_atencion.py
__file__ = 'snippets/ep09_atencion.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import os, sys, json
import torch
import torch.nn.functional as F
AQUI = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, AQUI)
from _cifar import cargar, normalizar
from _resnet import cargar_red
torch.set_grad_enabled(False)
X, y = cargar("test", n=1)
vit, _ = cargar_red(f"{AQUI}/../datos/ep09_vit_aug.pt")
capa = vit.encoder.layers[-1]                # el último bloque del ViT
z = vit.tokens(normalizar(X))
for c in vit.encoder.layers[:-1]:
    z = c(z)

In [7]:
h = capa.norm1(z)                            # (1, 65, 128): CLS + 64 patches
Wq, Wk, Wv = capa.self_attn.in_proj_weight.chunk(3)
bq, bk, bv = capa.self_attn.in_proj_bias.chunk(3)
H, dh = 4, 128 // 4                          # 4 heads de 32
Q = (h @ Wq.T + bq).view(65, H, dh).transpose(0, 1)   # (4, 65, 32)
K = (h @ Wk.T + bk).view(65, H, dh).transpose(0, 1)
A = F.softmax(Q @ K.transpose(1, 2) / dh ** 0.5, dim=-1)  # (4, 65, 65)

_, A_mod = capa.self_attn(h, h, h, average_attn_weights=False)
print("A:", tuple(A.shape), " dif. con PyTorch:",
      f"{(A - A_mod[0]).abs().max().item():.1e}")
cls = A[:, 0, 1:].mean(0).view(8, 8)          # qué patches mira el CLS
print("cada fila suma:", round(A[0, 0].sum().item(), 3))

A: (4, 65, 65)  dif. con PyTorch: 1.5e-07
cada fila suma: 1.0


In [8]:
# Verificación extra (corrió junto con el video)
# datos para las figuras del ep09 (se mezclan con los de ep09_posiciones)
ruta = os.path.join(AQUI, "..", "datos", "ep09_fig.json")
fig = json.load(open(ruta)) if os.path.exists(ruta) else {}
logits0 = (Q[0, 0] @ K[0, 1:].T) / dh ** 0.5            # head 0, CLS contra los 64 patches
top = logits0.argsort(descending=True)[:3].tolist() + logits0.argsort()[:3].tolist()
sub = logits0[top]
fig["cls_labels"] = [f"({j // 8},{j % 8})" for j in top]
fig["cls_vlabels"] = [f"v_{{{j // 8},{j % 8}}}" for j in top]
fig["cls_logits"] = [round(v, 2) for v in sub.tolist()]
fig["cls_probs"] = [round(v, 2) for v in F.softmax(sub, -1).tolist()]
fig["cls_mapa"] = [[round(100 * v, 1) for v in fila] for fila in cls.tolist()]
fig["clase"] = int(y[0])
fig["gato"] = [[round(v, 2) for v in fila] for fila in (0.299 * X[0, 0] + 0.587 * X[0, 1] + 0.114 * X[0, 2]).div(255).tolist()]
json.dump(fig, open(ruta, "w"))

### A qué patches mira el CLS

A la izquierda, la imagen: un gato sobre una tela azul. A la derecha, la atención del CLS en el último bloque, promediada sobre las cuatro heads: más claro, más atención. Casi nada va a la tela azul de abajo. La atención se reparte por la zona del gato y el fondo de arriba, con un punto brillante a la izquierda, sobre el borde de un objeto blanco. Ojo con la lectura: es una sola capa, de seis, y que el CLS mire un patch no prueba que la decisión dependa de él, igual que pasaba con la saliencia.

### La misma atención, en números (%)

Los mismos números. Si el CLS repartiera la atención pareja, cada patch tendría uno coma cinco por ciento. La última fila, y la mitad derecha de las filas cinco y seis, tienen dos décimas por patch: siete veces menos que lo parejo. En las tres primeras filas, casi todos pasan de uno coma cinco, hasta tres coma siete. Y el máximo, cinco coma cinco, está en la fila cinco, columna uno.

### Un bloque transformer

Cada bloque del ViT tiene la misma estructura que un bloque de un modelo de lenguaje. Una atención, donde los tokens intercambian información, y un MLP, que procesa cada token solo. Las dos salidas se suman al token, no lo reemplazan: es la conexión residual del episodio siete. La vía principal acumula correcciones bloque tras bloque. En el dibujo dice embedding y logits porque es el esquema genérico de un transformer; en el ViT, la entrada son los patches y la salida que se usa es la del CLS.

## Parte 3 · ViT contra ResNet

Mismos datos, mismas treinta epochs, sin y con data augmentation.

### Precisión de test

La comparación, en dos mil imágenes de test. Cuatro modelos: la ResNet de veinte capas del episodio siete, sin y con data augmentation, y el ViT, con los mismos datos y las mismas treinta epochs, cada uno con su optimizador. Precisión de test. ResNet: ochenta y cuatro coma dos. ViT: setenta y uno coma seis, doce puntos menos con el triple de parámetros. Con augmentation, los dos mejoran: la ResNet llega a ochenta y nueve coma uno, el ViT a ochenta coma tres. La augmentation le sirve más al ViT, nueve puntos, pero no alcanza para cerrar la distancia.

In [9]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep09_comparar.py
__file__ = 'snippets/ep09_comparar.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import os, sys
import torch
AQUI = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, AQUI)
from _cifar import cargar, normalizar
from _resnet import cargar_red, contar
from _redes import evaluar
torch.set_grad_enabled(False)
Xte, yte = cargar("test", n=2000)
Xte = normalizar(Xte)

In [10]:
modelos = [("ResNet-20", "ep07_res20"), ("ViT", "ep09_vit"),
           ("ResNet-20 + aug", "ep08_res20_aug"), ("ViT + aug", "ep09_vit_aug")]
for nombre, archivo in modelos:
    red, info = cargar_red(f"{AQUI}/../datos/{archivo}.pt")
    acc = evaluar(red, Xte, yte)             # 2.000 de test
    print(f"{nombre:16} {contar(red):>8,} parám.  test {acc:.1%}")

ResNet-20         272,474 parám.  test 84.2%


ViT               811,146 parám.  test 71.6%


ResNet-20 + aug   272,474 parám.  test 89.1%


ViT + aug         811,146 parám.  test 80.3%


### Precisión de test por epoch

La ResNet sin augmentation, punteada celeste, y con augmentation, llena. El ViT sin augmentation, punteado violeta: sube rápido al principio y se estanca cerca de setenta y dos; en train llega a cien por ciento, así que memoriza. Y el ViT con augmentation sigue subiendo hasta el final. A esta escala, con cincuenta mil imágenes, la red convolucional gana en todo el recorrido.

## Parte 4 · Lo que el ViT tiene que aprender

La convolución trae la vecindad de fábrica. El ViT la tiene que descubrir.

### ¿Aprendió que hay vecinos?

Por ejemplo, qué patch está al lado de cuál. Tomamos los embeddings posicionales aprendidos, sesenta y cuatro vectores de ciento veintiocho, normalizados. Coseno entre la posición de la fila tres, columna cuatro, y todas las demás. En el ViT entrenado, y en uno sin entrenar, como control. Entrenado: con el vecino de al lado, cero coma ochenta y cuatro; con la esquina de abajo a la izquierda, menos cero coma sesenta y dos. Sin entrenar: menos cero coma trece y menos cero coma cero cinco, ruido.

In [11]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep09_posiciones.py
__file__ = 'snippets/ep09_posiciones.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import os, sys
import torch
import torch.nn.functional as F
AQUI = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, AQUI)
from _resnet import cargar_red, ViTChica
torch.set_grad_enabled(False)
vit, _ = cargar_red(f"{AQUI}/../datos/ep09_vit_aug.pt")
torch.manual_seed(0)
al_azar = ViTChica()

In [12]:
def parecido(red, fila, col):
    E = F.normalize(red.pos[0, 1:], dim=-1)     # 64 posiciones × 128
    k = fila * 8 + col                          # el patch (fila, col)
    return (E @ E[k]).view(8, 8)                # coseno contra los 64

s = parecido(vit, 3, 4)
print("vecino de al lado:", round(s[3, 5].item(), 2),
      " lejos (fila 7, col 0):", round(s[7, 0].item(), 2))
s0 = parecido(al_azar, 3, 4)
print("sin entrenar, vecino:", round(s0[3, 5].item(), 2),
      " lejos:", round(s0[7, 0].item(), 2))

vecino de al lado: 0.84  lejos (fila 7, col 0): -0.62
sin entrenar, vecino: -0.13  lejos: -0.05


In [13]:
# Verificación extra (corrió junto con el video)
import json
ruta = os.path.join(AQUI, "..", "datos", "ep09_fig.json")
fig = json.load(open(ruta)) if os.path.exists(ruta) else {}
fig["pos_mapa"] = [[round(v, 2) for v in fila] for fila in s.tolist()]
json.dump(fig, open(ruta, "w"))

### Coseno entre la posición (3, 4) y las otras 63

El mapa completo de esos cosenos. La posición tres, cuatro vale uno, y alrededor se forma una mancha que se va apagando con la distancia, hasta valores negativos en las esquinas. Nadie le dijo al ViT que la imagen es una grilla de ocho por ocho: lo dedujo de los datos. La convolución, en cambio, tiene la vecindad escrita en su forma, desde el episodio dos. Eso que una arquitectura asume de antemano se llama inductive bias, sesgo inductivo, y es lo que el ViT no tiene.

> **Dosovitskiy et al., 2020 — An Image is Worth 16x16 Words** · [arXiv:2010.11929](https://arxiv.org/abs/2010.11929)
>
> **14M–300M imágenes** · En ImageNet sin regularización fuerte, «a few percentage points below ResNets of comparable size»; con 14M–300M imágenes, «large scale training trumps inductive bias».

El paper que presentó el ViT. Su título dice que una imagen vale dieciséis por dieciséis palabras, por el tamaño de patch. Entrenado en ImageNet, un dataset mediano para ellos, sin regularización fuerte, el ViT queda algunos puntos por debajo de ResNets de tamaño comparable, como acá. Pero con catorce a trescientos millones de imágenes, la escala le gana al inductive bias: su mejor modelo llega a ochenta y ocho coma cinco cinco por ciento en ImageNet.

## Resumiendo

- Patches de 4×4 → 64 tokens; patch embedding = conv con kernel = stride
- CLS + embeddings posicionales aprendidos: 65 tokens
- Atención: softmax(QKᵀ/√d)·V, cada token mira a todos
- Embeddings posicionales: aprendió la vecindad (0,84 vs. −0,62)
- 30 epochs, 50.000 imágenes: ViT 80,3 % vs. ResNet-20 89,1 %
- Próximo: unir imágenes y texto, CLIP